# CORAL Command-Line Tutorial

This tutorial shows how to run CORAL from the command line. See [README.md](README.md) for installation and
[OUTPUT_FORMAT.md](OUTPUT_FORMAT.md) for the output files. `coral <command> --help` lists every option.

---

## 1. Trio Pipeline (two sisters and an outgroup)

`run_trio` processes three taxa: an outgroup (the reference) and two sister species. It downloads the genomes from
NCBI, indexes the reference, fragments the sister genomes into overlapping pseudo-reads, aligns them to the reference,
extracts the mutations, normalizes the counts and writes plots and summary tables.

```bash
coral run_trio \
  --outgroup Saccharomyces_mikatae_IFO_1815 GCF_947241705.1 \
  --species Saccharomyces_paradoxus GCF_002079055.1 \
            Saccharomyces_cerevisiae_S288C GCF_000146045.2 \
  --output ../test_output \
  --suffix test
```

Key options:

- `--outgroup`: Reference species (name and NCBI accession) - **Required**
- `--species`: Two sister species (name1, acc1, name2, acc2) - **Required**
- `--output`: Output directory - **Required**
- `--suffix`: Optional suffix appended to the run_id
- `--divergence-time`: Divergence time in million years (for annualized substitution estimates)
- `--five-mer`: Also count the mutations by 5-mer context

---

## 2. Pair Pipeline (reference and target)

`run_pair` compares two genomes. With no outgroup the direction of a change is unknown, so differences are folded
into 52 unpolarized classes.

```bash
coral run_pair \
  --reference Saccharomyces_mikatae_IFO_1815 GCF_947241705.1 \
  --target Saccharomyces_cerevisiae_S288C GCF_000146045.2 \
  --output ../test_output
```

---

## 3. Multi-Species Pipeline (experimental)

`run_multi` processes several species, from a Newick tree or a species list. Results should be interpreted with
caution and are best used for exploratory analyses. It requires PHYLIP.

From a Newick tree (species and accession separated by `|`, ending with `;`):

```bash
coral run_multi \
  --newick-tree "(((Drosophila_melanogaster|GCF_000001215.4,Drosophila_sechellia|GCF_004382195.2),Drosophila_mauritiana|GCF_004382145.1),Drosophila_simulans|GCF_016746395.2);" \
  --outgroup Drosophila_simulans \
  --output ../test_output \
  --run-id drosophila_test
```

From a species list (a JSON array of `[name, accession]` pairs, quoted for the shell):

```bash
coral run_multi \
  --species-list '[["Drosophila_melanogaster","GCF_000001215.4"],["Drosophila_sechellia","GCF_004382195.2"],["Drosophila_mauritiana","GCF_004382145.1"],["Drosophila_simulans","GCF_016746395.2"]]' \
  --outgroup Drosophila_simulans \
  --output ../test_output \
  --run-id drosophila_test
```

`run_phylip` runs PHYLIP on the mutation matrix written by `run_multi`:

```bash
coral run_phylip \
  --df ../test_output/drosophila_test/matching_bases.csv.gz \
  --tree ../test_output/drosophila_test/annotated_tree.nwk \
  --mapping ../test_output/drosophila_test/species_mapping.json \
  --phylip-command dnapars \
  --prefix drosophila_phylip
```

---

## 4. Common Options

- `--cores N`: Number of CPU cores (default: all available)
- `--mapq N`: MAPQ threshold for keeping a pseudo-read without continuity support (default: 60)
- `--low-mapq N`: Minimum MAPQ for a pseudo-read rescued by an overlapping neighbour (default: 1)
- `--no-continuity`: Keep only pseudo-reads with MAPQ ≥ `--mapq`
- `--indel-window K`: Also exclude sites within K bp of an indel (default: 1, only the window itself)
- `--repeat-mask`: Skip mostly repetitive pseudo-reads and masked reference positions (WindowMasker by default)
- `--no-cache`: Rebuild stages instead of reusing existing files (they are reused whatever the options)
- `--no-cleanup`: Keep the genomes, BAMs and other intermediate files
- `--quiet`: Disable verbose logging

BWA-MEM is the default aligner. Another one is chosen with `--aligner-name bwa-mem2`, `minimap2` or `bbmap`, or given
as a command with placeholders `{ref}`, `{fq}` and `{cores}`. The command must write SAM to standard output, one
primary record per pseudo-read in input order. An aligner that needs an index also gets its index command
(`--index-cmd`), which CORAL runs once on the reference:

```bash
coral run_trio ... --index-cmd "minimap2 -x sr -k 15 -d {ref}.k15.mmi {ref}" \
                   --aligner-cmd "minimap2 -t {cores} -ax sr {ref}.k15.mmi {fq}"
```

Other aligners may use a different MAPQ scale, so adjust `--mapq` and `--low-mapq`.